In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import accuracy_score, roc_auc_score
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm


# Load data
tropomi_combined_dropna = pd.read_csv('/net/fs06/d3/rzhuang/TROPOMI_world/data/Run_3/updated_tropomi_emissions_full_variables_with_fuel.csv')

# Get feature names
FEATURES = [
    'annual_nox_emission', 'surface_altitude', 'surface_altitude_precision',
    'surface_classification', 'surface_pressure', 'surface_albedo',
    'surface_albedo_nitrogendioxide_window', 'cloud_pressure_crb',
    'cloud_fraction_crb', 'cloud_albedo_crb', 'scene_albedo',
    'apparent_scene_pressure', 'snow_ice_flag', 'aerosol_index_354_388', 
    'scaled_small_pixel_variance', 
    'sensor_altitude', 'sensor_azimuth_angle', 'sensor_zenith_angle', 
    'solar_azimuth_angle', 'solar_zenith_angle', 'wind_speed', 't2m', 'tisr', 'tcwv',
    'primary_fuel_type',
]


feature_names = FEATURES
print(f"Number of features to analyze: {len(FEATURES)}")

le = LabelEncoder()
tropomi_combined_dropna['primary_fuel_type'] = le.fit_transform(tropomi_combined_dropna['primary_fuel_type'])
# Extract X with the specified features
X = tropomi_combined_dropna[FEATURES].values.astype(np.float32)
y = tropomi_combined_dropna["plume_label"].astype(int).values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
ros = RandomOverSampler(random_state=42)
X_tr_bal, y_tr_bal = ros.fit_resample(X_train, y_train)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr_bal)
X_te_s = scaler.transform(X_test)

# Model definition
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

# Load trained model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MLP(X_tr_s.shape[1]).to(device)
model.load_state_dict(torch.load('/net/fs06/d3/rzhuang/TROPOMI_world/model/best_model_all_data_filtered_no_stats.pt'))
model.eval()

# Calculate original AUC
test_ds = TensorDataset(torch.from_numpy(X_te_s), torch.from_numpy(y_test))
test_dl = DataLoader(test_ds, batch_size=32)

def calculate_auc(model, data_loader, y_true):
    probs = []
    with torch.no_grad():
        for xb, _ in data_loader:
            xb = xb.to(device)
            probs.extend(torch.sigmoid(model(xb)).cpu().numpy())
    return roc_auc_score(y_true, probs)

orig_auc = calculate_auc(model, test_dl, y_test)
print(f"Original AUC: {orig_auc:.4f}")

# Permutation importance
n_features = X_te_s.shape[1]
importances = np.zeros(n_features)
n_repeats = 5  # Number of times to repeat permutation

print("\nCalculating permutation importance...")
for i in tqdm(range(n_features)):
    scores = []
    for _ in range(n_repeats):
        # Create a copy of test data
        X_perm = X_te_s.copy()
        # Permute feature i
        np.random.shuffle(X_perm[:, i])
        # Create new dataloader with permuted data
        perm_ds = TensorDataset(torch.from_numpy(X_perm), torch.from_numpy(y_test))
        perm_dl = DataLoader(perm_ds, batch_size=32)
        # Calculate AUC with permuted feature
        perm_auc = calculate_auc(model, perm_dl, y_test)
        # Importance is the drop in AUC
        scores.append(orig_auc - perm_auc)
    importances[i] = np.mean(scores)

# Create results dataframe
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

print("\nTop 20 Most Important Features:")
print(importance_df.head(20))

# Save results
importance_df.to_csv('/net/fs06/d3/rzhuang/TROPOMI_world/data/Run_3/feature_importance_permutation.csv', index=False)

# Visualize
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
top_features = importance_df.head(20)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Permutation Importance (AUC drop)')
plt.title('Top 20 Most Important Features')
plt.tight_layout()
plt.savefig('/net/fs06/d3/rzhuang/TROPOMI_world/data/Run_3/feature_importance_plot.png', dpi=300)
plt.close()

print(f"\nResults saved to feature_importance_permutation.csv")
print(f"Plot saved to feature_importance_plot.png")

Number of features to analyze: 25
Original AUC: 0.6998

Calculating permutation importance...


100%|██████████| 25/25 [16:58<00:00, 40.74s/it]



Top 20 Most Important Features:
                                  feature  importance
0                     annual_nox_emission    0.101393
1                        surface_altitude    0.028724
17                    sensor_zenith_angle    0.026569
24                      primary_fuel_type    0.025662
15                        sensor_altitude    0.022861
5                          surface_albedo    0.021144
6   surface_albedo_nitrogendioxide_window    0.019787
21                                    t2m    0.015884
20                             wind_speed    0.015396
3                  surface_classification    0.013996
19                     solar_zenith_angle    0.012279
2              surface_altitude_precision    0.011662
22                                   tisr    0.011296
4                        surface_pressure    0.009958
10                           scene_albedo    0.008157
23                                   tcwv    0.007470
13                  aerosol_index_354_388    0.00

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
World permutation importance with STRICT full-frame dropna base
+ Interference-zone filtering using global plants/cities.
Uses plants file with columns: ID, latitude, longitude, annual_nox_emission.
"""

import os, time, json
import numpy as np
import pandas as pd
from typing import List, Any, Optional, Dict

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import roc_auc_score, average_precision_score

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.neighbors import BallTree
from haversine import haversine
from math import radians, log10

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm import tqdm

# -------------------------
# Paths / Config
# -------------------------
DATA_TROPOMI = '/net/fs06/d3/rzhuang/TROPOMI_world/data/Run_3/updated_tropomi_emissions_full_variables_with_fuel.csv'
DATA_PLANTS  = '/net/fs06/d3/rzhuang/TROPOMI_world/data/power_plant_location/power_plants_with_combined_nearby_stats.csv'
DATA_CITIES  = '/net/fs06/d3/rzhuang/TROPOMI_world/data/worldcities.csv'
MODEL_PATH   = '/net/fs06/d3/rzhuang/TROPOMI_world/model/best_model_all_data_filtered_no_stats.pt'
OUT_ROOT     = '/net/fs06/d3/rzhuang/TROPOMI_world/data/Run_3/no_interference'
os.makedirs(OUT_ROOT, exist_ok=True)

BATCH_SIZE   = 4096
TEST_SIZE    = 0.20
RANDOM_STATE = 42
N_REPEATS    = 5
TOPK_PLOT    = 20
NUM_WORKERS  = 0
PIN_MEMORY   = torch.cuda.is_available()

# Interference constants
EARTH_RADIUS_KM      = 6371
PLANT_RADIUS_BASE_KM = 20.0
CITY_POP_THRESHOLD   = 200000
CITY_RADIUS_SCALE    = 9.0
CITY_RADIUS_BASE_KM  = 10.0
CITY_RADIUS_MAX_KM   = 90.0

FEATURES: List[str] = [
    'annual_nox_emission', 'surface_altitude', 'surface_altitude_precision',
    'surface_classification', 'surface_pressure', 'surface_albedo',
    'surface_albedo_nitrogendioxide_window', 'cloud_pressure_crb',
    'cloud_fraction_crb', 'cloud_albedo_crb', 'scene_albedo',
    'apparent_scene_pressure', 'snow_ice_flag', 'aerosol_index_354_388',
    'scaled_small_pixel_variance',
    'sensor_altitude', 'sensor_azimuth_angle', 'sensor_zenith_angle',
    'solar_azimuth_angle', 'solar_zenith_angle', 'wind_speed', 't2m', 'tisr', 'tcwv',
    'primary_fuel_type',
]

def log(m): print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {m}", flush=True)
def set_seed(s=42):
    np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def safe_roc_auc(y, p):
    try:
        if len(np.unique(y)) < 2: return float('nan')
        return float(roc_auc_score(y, p))
    except Exception:
        return float('nan')

# -------------------------
# Model
# -------------------------
class MLP(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(1)

def load_model(dim, path, device):
    m = MLP(dim).to(device)
    m.load_state_dict(torch.load(path, map_location=device))
    m.eval()
    return m

def predict_probs(model: nn.Module, X: np.ndarray, batch_size=BATCH_SIZE) -> np.ndarray:
    ds = TensorDataset(torch.from_numpy(X))
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    probs = np.empty((len(X),), dtype=np.float32); i0 = 0
    with torch.inference_mode():
        for (xb,) in dl:
            xb = xb.to(next(model.parameters()).device, non_blocking=True, dtype=torch.float32)
            p = torch.sigmoid(model(xb)).detach().cpu().numpy().reshape(-1)
            n = len(p); probs[i0:i0+n] = p; i0 += n
    return probs

# -------------------------
# Interference (plants: ID/latitude/longitude/annual_nox_emission)
# -------------------------
def identify_plants_in_interference_zones_world(plants_df, cities_df, plant_subset_ids=None):
    """
    Identifies which plants are in interference zones of other plants or cities.
    Adapted for world data structure.
    """
    print("Identifying plants in interference zones (global)...")
    
    # Filter to subset if provided
    if plant_subset_ids is not None:
        plants_df = plants_df[plants_df['ID'].isin(plant_subset_ids)].copy()
        print(f"Checking interference for {len(plants_df)} plants in subset")
    
    # Ensure we have the required columns
    if 'nox_emis_ty' in plants_df.columns:
        plants_df['annual_nox_emission'] = plants_df['nox_emis_ty']
    else:
        print("Warning: No annual_nox_emission column found, using index as proxy")
        plants_df['annual_nox_emission'] = -plants_df.index
    
    # Build BallTree for plants
    plants_df['lat_rad'] = np.radians(plants_df['latitude'])
    plants_df['lon_rad'] = np.radians(plants_df['longitude'])
    plant_tree = BallTree(plants_df[['lat_rad', 'lon_rad']].values, metric='haversine')
    
    # Build BallTree for cities
    cities_filtered = cities_df[cities_df['population'] >= CITY_POP_THRESHOLD].copy()
    cities_filtered['lat_rad'] = np.radians(cities_filtered['latitude'])
    cities_filtered['lon_rad'] = np.radians(cities_filtered['longitude'])
    
    if len(cities_filtered) > 0:
        city_tree = BallTree(cities_filtered[['lat_rad', 'lon_rad']].values, metric='haversine')
    else:
        city_tree = None
        print("Warning: No cities above population threshold")
    
    interfered_plants = set()
    
    for idx, target_plant in tqdm(plants_df.iterrows(), total=len(plants_df), desc="Checking interference"):
        target_id = target_plant['ID']
        target_lat = target_plant['latitude']
        target_lon = target_plant['longitude']
        target_emissions = target_plant.get('annual_nox_emission', 0)
        
        if pd.isna(target_lat) or pd.isna(target_lon):
            continue
            
        target_coords_rad = np.array([[radians(target_lat), radians(target_lon)]])
        
        # Check interference from other plants
        search_radius = PLANT_RADIUS_BASE_KM / EARTH_RADIUS_KM
        nearby_plant_indices = plant_tree.query_radius(target_coords_rad, r=search_radius)[0]
        
        for plant_idx in nearby_plant_indices:
            if plant_idx == idx:  # Skip self
                continue
            source_plant = plants_df.iloc[plant_idx]
            source_emissions = source_plant.get('annual_nox_emission', 0)
            
            # Only interfered if source has higher emissions
            if pd.notna(source_emissions) and source_emissions > target_emissions:
                distance_km = haversine(
                    (target_lat, target_lon),
                    (source_plant['latitude'], source_plant['longitude'])
                )
                if distance_km < PLANT_RADIUS_BASE_KM:
                    interfered_plants.add(target_id)
                    break
        
        # Check interference from cities
        if city_tree and target_id not in interfered_plants:
            search_radius = CITY_RADIUS_MAX_KM / EARTH_RADIUS_KM
            nearby_city_indices = city_tree.query_radius(target_coords_rad, r=search_radius)[0]
            
            for city_idx in nearby_city_indices:
                source_city = cities_filtered.iloc[city_idx]
                population = source_city['population']
                
                # Calculate city interference radius
                radius = CITY_RADIUS_BASE_KM + (log10(max(1, population)) * CITY_RADIUS_SCALE)
                interference_radius_km = min(radius, CITY_RADIUS_MAX_KM)
                
                distance_km = haversine(
                    (target_lat, target_lon),
                    (source_city['latitude'], source_city['longitude'])
                )
                
                if distance_km < interference_radius_km:
                    interfered_plants.add(target_id)
                    break
    
    print(f"Found {len(interfered_plants)} plants in interference zones out of {len(plants_df)} checked")
    return interfered_plants

# -------------------------
# Permutation runner
# -------------------------
def run_perm_importance_for_dataset(dataset_name: str,
                                    df: pd.DataFrame,
                                    out_dir: str,
                                    device: torch.device) -> Dict[str, Any]:
    os.makedirs(out_dir, exist_ok=True)

    missing = [f for f in FEATURES if f not in df.columns]
    if missing: raise KeyError(f"[{dataset_name}] Missing features: {missing}")

    # encode fuel if needed
    if not (pd.api.types.is_integer_dtype(df['primary_fuel_type']) or pd.api.types.is_float_dtype(df['primary_fuel_type'])):
        le = LabelEncoder()
        df = df.copy()
        df['primary_fuel_type'] = le.fit_transform(df['primary_fuel_type'])

    # strict base already applied; still ensure required cols exist
    need = set(FEATURES) | {'plume_label'}
    dfx = df.dropna(subset=list(need)).copy()
    if len(dfx) == 0:
        log(f"[{dataset_name}] empty after NA drop. Skipping.")
        return {"n": 0}

    X = dfx[FEATURES].values.astype(np.float32)
    y = dfx['plume_label'].astype(int).values

    # split/oversample/scale
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
    ros = RandomOverSampler(random_state=RANDOM_STATE)
    X_trb, y_trb = ros.fit_resample(X_tr, y_tr)
    scaler = StandardScaler()
    X_trs = scaler.fit_transform(X_trb)
    X_tes = scaler.transform(X_te)

    # model
    model = load_model(X_trs.shape[1], MODEL_PATH, device)
    y_prob = predict_probs(model, X_tes)
    auc = safe_roc_auc(y_te, y_prob)
    pr  = float(average_precision_score(y_te, y_prob))
    log(f"[{dataset_name}] Holdout: ROC-AUC={auc:.4f} | PR-AUC={pr:.4f} | N={len(y_te)}")

    # permutation importance
    rng = np.random.default_rng(RANDOM_STATE)
    importances = np.zeros(X_tes.shape[1], dtype=np.float32)
    log(f"[{dataset_name}] Permutation importance: {X_tes.shape[1]} features, repeats={N_REPEATS}...")
    for i in range(X_tes.shape[1]):
        drops = []
        for _ in range(N_REPEATS):
            Xp = X_tes.copy()
            rng.shuffle(Xp[:, i])
            p = predict_probs(model, Xp)
            drops.append(auc - safe_roc_auc(y_te, p))
        importances[i] = float(np.mean(drops))

    imp_df = pd.DataFrame({'feature': FEATURES, 'importance_auc_drop': importances}) \
             .sort_values('importance_auc_drop', ascending=False).reset_index(drop=True)
    csv_path = os.path.join(out_dir, f'feature_importance_permutation_{dataset_name}.csv')
    imp_df.to_csv(csv_path, index=False); log(f"[{dataset_name}] Saved CSV: {csv_path}")

    top = imp_df.head(TOPK_PLOT).iloc[::-1]
    plt.figure(figsize=(10, 8))
    plt.barh(range(len(top)), top['importance_auc_drop'])
    plt.yticks(range(len(top)), top['feature'])
    plt.xlabel('Permutation Importance (AUC drop)')
    plt.title(f'Top {TOPK_PLOT} Features — {dataset_name}')
    plt.tight_layout()
    png_path = os.path.join(out_dir, f'feature_importance_permutation_top{TOPK_PLOT}_{dataset_name}.png')
    plt.savefig(png_path, dpi=300); plt.close()
    log(f"[{dataset_name}] Saved plot: {png_path}")

    return {"n_rows_after_na": int(len(dfx)), "n_holdout": int(len(y_te)),
            "auc": float(auc), "pr_auc": float(pr), "csv": csv_path, "png": png_path}

# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    set_seed(RANDOM_STATE)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # load metadata
    log(f"Loading plants: {DATA_PLANTS}")
    plants = pd.read_csv(DATA_PLANTS)
    # keep any emission-based sorting if desired
    if 'annual_nox_emission' in plants.columns:
        plants = plants.sort_values('annual_nox_emission', ascending=False)

    log(f"Loading cities: {DATA_CITIES}")
    cities = pd.read_csv(DATA_CITIES)

    # load TROPOMI and build STRICT base
    log(f"Loading world TROPOMI: {DATA_TROPOMI}")
    df_raw = pd.read_csv(DATA_TROPOMI)
    df_base = df_raw.dropna().copy()

    # encode fuel AFTER drop
    if not (pd.api.types.is_integer_dtype(df_base['primary_fuel_type']) or
            pd.api.types.is_float_dtype(df_base['primary_fuel_type'])):
        le_tmp = LabelEncoder()
        df_base['primary_fuel_type'] = le_tmp.fit_transform(df_base['primary_fuel_type'])
        log("Encoded 'primary_fuel_type' (post dropna).")

    # IDs present in strict base
    if 'location' not in df_base.columns:
        raise KeyError("'location' column missing in world TROPOMI file.")
    all_ids = df_base['location'].dropna().unique().tolist()

    # align plant metadata by ID
    plants_for_all = plants[plants['ID'].isin(all_ids)].copy()
    log(f"Strict base: rows={len(df_base)} | unique plants={len(all_ids)} | matched plants={len(plants_for_all)}")

    # interference set
    interfered_all = identify_plants_in_interference_zones_world(plants_for_all, cities, plant_subset_ids=all_ids)

    # datasets from SAME strict base
    unfiltered_all = df_base.copy()
    filtered_all   = df_base[~df_base['location'].isin(interfered_all)].copy()

    # stats
    log("Dataset stats (strict base):")
    total_plants = len(all_ids)
    log(f"  ALL: plants={total_plants}, interfered={len(interfered_all)}, clean={total_plants - len(interfered_all)}")
    log(f"  Original observations: {len(unfiltered_all)}")
    log(f"  Filtered observations: {len(filtered_all)} (removed {len(unfiltered_all) - len(filtered_all)})")

    # save interference info
    info = {
        "world_all": {
            "total_plants": total_plants,
            "interfered_plants": len(interfered_all),
            "clean_plants": total_plants - len(interfered_all),
            "interfered_plant_ids": list(map(str, interfered_all))
        }
    }
    with open(os.path.join(OUT_ROOT, 'interference_info_perm_importance_world.json'), 'w') as f:
        json.dump(info, f, indent=2)

    # output dirs
    out_unfiltered = os.path.join(OUT_ROOT, 'unfiltered')
    out_filtered   = os.path.join(OUT_ROOT, 'filtered')
    os.makedirs(out_unfiltered, exist_ok=True)
    os.makedirs(out_filtered, exist_ok=True)

    # run permutation importance
    results = {}
    results['unfiltered_world'] = run_perm_importance_for_dataset('unfiltered_world', unfiltered_all, out_unfiltered, device)
    results['filtered_world']   = run_perm_importance_for_dataset('filtered_world',   filtered_all,   out_filtered,   device)

    with open(os.path.join(OUT_ROOT, 'perm_importance_summary_world.json'), 'w') as f:
        json.dump(results, f, indent=2)

    log("Done.")

[2025-08-28 14:17:02] Loading plants: /net/fs06/d3/rzhuang/TROPOMI_world/data/power_plant_location/power_plants_with_combined_nearby_stats.csv
[2025-08-28 14:17:02] Loading cities: /net/fs06/d3/rzhuang/TROPOMI_world/data/worldcities.csv
[2025-08-28 14:17:02] Loading world TROPOMI: /net/fs06/d3/rzhuang/TROPOMI_world/data/Run_3/updated_tropomi_emissions_full_variables_with_fuel.csv
[2025-08-28 14:17:17] Encoded 'primary_fuel_type' (post dropna).
[2025-08-28 14:17:17] Strict base: rows=875580 | unique plants=6000 | matched plants=6000
Identifying plants in interference zones (global)...
Checking interference for 6000 plants in subset


Checking interference: 100%|██████████| 6000/6000 [00:03<00:00, 1845.68it/s]


Found 4935 plants in interference zones out of 6000 checked
[2025-08-28 14:17:21] Dataset stats (strict base):
[2025-08-28 14:17:21]   ALL: plants=6000, interfered=4935, clean=1065
[2025-08-28 14:17:21]   Original observations: 875580
[2025-08-28 14:17:21]   Filtered observations: 161118 (removed 714462)
[2025-08-28 14:17:26] [unfiltered_world] Holdout: ROC-AUC=0.6957 | PR-AUC=0.1955 | N=175116
[2025-08-28 14:17:26] [unfiltered_world] Permutation importance: 25 features, repeats=5...
[2025-08-28 14:19:59] [unfiltered_world] Saved CSV: /net/fs06/d3/rzhuang/TROPOMI_world/data/Run_3/no_interference/unfiltered/feature_importance_permutation_unfiltered_world.csv
[2025-08-28 14:19:59] [unfiltered_world] Saved plot: /net/fs06/d3/rzhuang/TROPOMI_world/data/Run_3/no_interference/unfiltered/feature_importance_permutation_top20_unfiltered_world.png
[2025-08-28 14:20:00] [filtered_world] Holdout: ROC-AUC=0.8157 | PR-AUC=0.7718 | N=32224
[2025-08-28 14:20:00] [filtered_world] Permutation importance

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import accuracy_score, roc_auc_score
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm

# Load data
tropomi_combined_dropna = pd.read_csv('/net/fs06/d3/rzhuang/TROPOMI_world/data/Run_3/updated_tropomi_emissions_full_variables_with_fuel.csv')

# Get feature names
FEATURES = [
    'annual_nox_emission', 'surface_altitude', 'surface_altitude_precision',
    'surface_classification', 'surface_pressure', 'surface_albedo',
    'surface_albedo_nitrogendioxide_window', 'cloud_pressure_crb',
    'cloud_fraction_crb', 'cloud_albedo_crb', 'scene_albedo',
    'apparent_scene_pressure', 'snow_ice_flag', 'aerosol_index_354_388', 
    'scaled_small_pixel_variance', 'tropospheric_NO2_column_number_density', 
    'sensor_altitude', 'sensor_azimuth_angle', 'sensor_zenith_angle', 
    'solar_azimuth_angle', 'solar_zenith_angle', 'wind_speed', 't2m', 'tisr', 'tcwv',
    'no2_mean_radius', 'no2_std_radius', 'no2_frac_valid_radius', 'primary_fuel_type',
]

feature_names = FEATURES
print(f"Number of features to analyze: {len(FEATURES)}")

le = LabelEncoder()
tropomi_combined_dropna['primary_fuel_type'] = le.fit_transform(tropomi_combined_dropna['primary_fuel_type'])
# Extract X with the specified features
X = tropomi_combined_dropna[FEATURES].values.astype(np.float32)
y = tropomi_combined_dropna["plume_label"].astype(int).values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
ros = RandomOverSampler(random_state=42)
X_tr_bal, y_tr_bal = ros.fit_resample(X_train, y_train)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr_bal)
X_te_s = scaler.transform(X_test)

# Model definition
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

# Load trained model (the one trained without nearby features)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MLP(X_tr_s.shape[1]).to(device)
LR = 1e-5
model.load_state_dict(torch.load(f'/net/fs06/d3/rzhuang/TROPOMI_world/model/best_0.0005_features_no_nearby.pt'))
model.eval()

# Calculate original AUC
test_ds = TensorDataset(torch.from_numpy(X_te_s), torch.from_numpy(y_test))
test_dl = DataLoader(test_ds, batch_size=32)

def calculate_auc(model, data_loader, y_true):
    probs = []
    with torch.no_grad():
        for xb, _ in data_loader:
            xb = xb.to(device)
            probs.extend(torch.sigmoid(model(xb)).cpu().numpy())
    return roc_auc_score(y_true, probs)

orig_auc = calculate_auc(model, test_dl, y_test)
print(f"\nOriginal AUC: {orig_auc:.4f}")

# Permutation importance
n_features = X_te_s.shape[1]
importances = np.zeros(n_features)
n_repeats = 5  # Number of times to repeat permutation

print("\nCalculating permutation importance...")
for i in tqdm(range(n_features)):
    scores = []
    for _ in range(n_repeats):
        # Create a copy of test data
        X_perm = X_te_s.copy()
        # Permute feature i
        np.random.shuffle(X_perm[:, i])
        # Create new dataloader with permuted data
        perm_ds = TensorDataset(torch.from_numpy(X_perm), torch.from_numpy(y_test))
        perm_dl = DataLoader(perm_ds, batch_size=32)
        # Calculate AUC with permuted feature
        perm_auc = calculate_auc(model, perm_dl, y_test)
        # Importance is the drop in AUC
        scores.append(orig_auc - perm_auc)
    importances[i] = np.mean(scores)

# Create results dataframe
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

print("\nTop 20 Most Important Features:")
print(importance_df.head(20))

# Save results
importance_df.to_csv('/net/fs06/d3/rzhuang/TROPOMI_world/data/Run_3/feature_importance_permutation_no_nearby.csv', index=False)

# Visualize
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
top_features = importance_df.head(20)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Permutation Importance (AUC drop)')
plt.title('Top 20 Most Important Features - World (No Nearby Features)')
plt.tight_layout()
plt.savefig('/net/fs06/d3/rzhuang/TROPOMI_world/data/Run_3/feature_importance_plot_no_nearby.png', dpi=300)
plt.close()

print(f"\nResults saved to feature_importance_permutation_no_nearby.csv")
print(f"Plot saved to feature_importance_plot_no_nearby.png")

Number of features to analyze: 29

Original AUC: 0.9310

Calculating permutation importance...


100%|██████████| 29/29 [11:39<00:00, 24.12s/it]



Top 20 Most Important Features:
                                   feature  importance
4                         surface_pressure    0.141221
1                         surface_altitude    0.135795
0                      annual_nox_emission    0.133027
25                         no2_mean_radius    0.094181
16                         sensor_altitude    0.085043
28                       primary_fuel_type    0.084352
26                          no2_std_radius    0.081839
6    surface_albedo_nitrogendioxide_window    0.060181
3                   surface_classification    0.049968
5                           surface_albedo    0.044655
2               surface_altitude_precision    0.041625
15  tropospheric_NO2_column_number_density    0.034876
20                      solar_zenith_angle    0.027471
12                           snow_ice_flag    0.021817
23                                    tisr    0.021787
22                                     t2m    0.018769
10                            sc